# SETUP

In [ ]:
import itertools
import math
import numpy as np
import os
import pandas as pd
import skimage.io

from datetime import datetime
from openpyxl import load_workbook
from pathlib import Path
from scipy.ndimage import gaussian_filter

import resource

## INPUT

In [ ]:
dir_input = "INPUT FOLDER"

### PREFERENCES

In [ ]:
distance_unit = "µm"
dist_per_px = 1 / 1
quantification_regions = 6
output_raw_values = False

## INPUT CHECK

In [ ]:
if type(dir_input) != str:
    raise TypeError("dir_input must be a string.")

if type(distance_unit) != str:
    distance_unit = "pixels"
    print("distance_unit must be a string; using default value (pixels).")

#dist_per_px
try:
    dist_per_px = float(dist_per_px)
    if not dist_per_px > 0:
        dist_per_px = 1
        print("dist_per_px must be greater than 0; using default value (1). Distance units = pixels.")
except:
    dist_per_px = 1
    print("dist_per_px must be greater than 0; using default value (1). Distance units = pixels.")

#quantification_regions
try:
    quantification_regions_f = float(quantification_regions)
    quantification_regions = float(round(quantification_regions))
    if quantification_regions < 1:
        quantification_regions = 6
        print("quantification_regions must be 1 or greater; using default value (6).")
    elif quantification_regions == 1:
        print("Caution: are you sure you want to segment data using a single \"region\"?")
    elif quantification_regions != quantification_regions_f:
        print(f'quantification_regions was rounded from {quantification_regions_f} to {quantification_regions}')
    quantification_regions = int(quantification_regions)
except:
    quantification_regions = 6
    print("quantification_regions must be 1 or greater; using default value (6).")

#output_raw_values
if type(output_raw_values) != bool:
    output_raw_values = False
    print("output_raw_values must be True or False; using default value (False).")

In [ ]:
region_fraction = 1 / quantification_regions
region_index = [f'Region {x + 1}' for x in range(quantification_regions)] + ["Overall"]
now = datetime.now() 
dt_string = now.strftime("%Y-%m-%d_%H%M%S")

# FUNCTIONS

In [ ]:
def filter_hidden(some_list):
    new_list = sorted([x for x in some_list if "._" not in x])
    if len(new_list) < len(some_list):
        print(f'Detected {len(some_list) - len(new_list)} hidden files in the input directory\n'
              + f'Hidden image files have been excluded from analysis.')
    return(new_list)

In [ ]:
def summary():
    analysis_parameters = f'Analysis parameters...\n\n'
    analysis_parameters += f'Regions (regions): {quantification_regions}\n'
    summary = open(f'{outputdir}Analysis parameters for {dir_input}.txt', "w")
    summary.write(analysis_parameters)
    summary.close()

In [ ]:
def tidy_columns(df_raw):
    df = df_raw.copy()
    for column in df.columns:
        if column in ["ImageNumber",
                      "ObjectNumber",
                      "Metadata_Channel",
                      "Metadata_Frame",
                      "Metadata_Processing",
                      "Metadata_Series"]:
            df.drop([column], axis = 1, inplace = True)
    df.columns = [x.split("Metadata_")[-1] for x in df.columns]
    df['Distance_Minimum_Spheroid'] = (1 - (df['Distance_Minimum_Spheroid'] / r_max)).clip(lower = 0.0001)
    df = df.rename(columns = {'Intensity_IntegratedIntensity_Masked_Target': f'Intensity_IntegratedIntensity_Masked_{target}',
                                              'Distance_Minimum_Spheroid': 'Frac_Dist_Core'})
    intensity_cols = [x for x in df.columns if "Intensity_IntegratedIntensity_Masked_" in x]
    image_channels = [x.split("_")[-1] for x in intensity_cols]
    df = df.rename(columns = dict(zip(intensity_cols, image_channels)))
    return(df, image_channels)

In [ ]:
def try_dir(some_dir):
    if not os.path.exists(some_dir):
        os.makedirs(some_dir)

# FILE PARSE

In [ ]:
path_files = sorted([str(path_file) for path_file in Path(dir_input).rglob("*.csv")])
files = filter_hidden(path_files)
file_details = [Path(file_name).stem.split("_") for file_name in path_files]
celltypes = sorted(set([details[0] for details in file_details]))
targets = sorted(set([details[2] for details in file_details]))
all_dfs = [pd.read_csv(df) for df in path_files]
info_and_dfs = list(zip(file_details, all_dfs))

# MAIN

In [ ]:
for celltype in celltypes:
    celltype_flist = [x for x in file_details if x[0] == celltype]
    if not celltype_flist:
        continue
    all_channels_of_interest = []
    targets = sorted(set(details[2] for details in celltype_flist))
    celltype_df_datasets = []
    for target in targets:
        target_flist = [x for x in celltype_flist if x[2] == target]
        if not target_flist:
            continue
        celltype_target_dfs = []
        for image_details in target_flist:
            image_csv = f'{"_".join(image_details)}.csv'
            image_data = pd.read_csv(Path(dir_input, image_csv))
            r_max = image_data['Distance_Minimum_Spheroid'].max()
            working_df, image_channels = tidy_columns(image_data)
            region_info_headers = ["Objects", f'Mean distance {distance_unit}', "Region area (px)"]
            region_lists = []
            for channel in image_channels:
                region_info_headers.append(f'{channel}')
            for i in range(quantification_regions):
                lower_dist_limit = region_fraction * i
                upper_dist_limit = region_fraction * (i + 1)
                region_slice = working_df[(lower_dist_limit < working_df['Frac_Dist_Core']) & (working_df['Frac_Dist_Core'] <= upper_dist_limit)]
                mean_dist_region = region_slice['Frac_Dist_Core'].mean() * r_max * dist_per_px
                sum_area_region = region_slice['AreaShape_Area'].sum()
                region_data = [len(region_slice.index), mean_dist_region, sum_area_region]
                for channel in image_channels:
                    region_data.append(region_slice[f'{channel}'].sum())
                region_lists.append(region_data)
            region_df = pd.DataFrame(region_lists, columns = region_info_headers)
            overall = pd.Series([region_df[x].sum() for x in region_df.columns])
            region_df.loc[len(region_df.index)] = dict(zip(region_info_headers, overall))
            region_df.index = region_index
            region_df.loc["Overall", f'Mean distance {distance_unit}'] = working_df['Frac_Dist_Core'].mean() * r_max * dist_per_px
            region_df[image_channels] = (region_df[image_channels].T / region_df.T.loc['Region area (px)']).T
            region_df.columns = [f'Mean {x}' if x in image_channels else x for x in region_df.columns]
            celltype_target_dfs.append(region_df.copy())
            all_channels_of_interest.extend(image_channels)
        celltype_target_values_dfs = []
        for channel in image_channels:
            values_lists = [x[f'Mean {channel}'] for x in celltype_target_dfs]
            values_df = pd.DataFrame(values_lists)
            midx = pd.MultiIndex.from_arrays([[celltype] * len(celltype_target_dfs),
                                              [f'Mean {channel}'] * len(celltype_target_dfs),
                                               pd.Series(range(len(celltype_target_dfs))) + 1],
                                               names = ["Celltype",
                                                        "Channel",
                                                        "Image no."])
            values_df.index = midx
            celltype_target_values_dfs.append(values_df) # per channel
        celltype_df_datasets.append((celltype_target_values_dfs, target)) # targets
    all_channels_of_interest = sorted(set(all_channels_of_interest))
    all_channels_of_interest = str(all_channels_of_interest).translate({ord(c): None for c in "'"})[:230]
    if len(celltypes) > 1:
        dir_output = Path("output", f'{dt_string} - {dir_input} - normalized spatial signal analysis ({quantification_regions} regions) ')
        out_name = f'{celltype} {all_channels_of_interest}.xlsx'
    else:
        dir_output = Path("output")
        out_name = f'{dt_string} - {dir_input} - normalized spatial signal analysis ({quantification_regions} regions) {celltype} {all_channels_of_interest}.xlsx'
    try_dir(dir_output)
    blank_df = pd.DataFrame()
    with pd.ExcelWriter(Path(dir_output, out_name)) as writer:
        blank_df.to_excel(writer)
    with pd.ExcelWriter(Path(dir_output, out_name), mode = "a", engine = "openpyxl", if_sheet_exists = "overlay") as writer:
        for celltype_target_dfset in celltype_df_datasets:
            row_tracker = 0
            values_dfs = celltype_target_dfset[0]
            target_name = celltype_target_dfset[1]
            for values_df in values_dfs:
                if not output_raw_values:
                    values_df = values_df / values_df['Region 1'].mean()
                values_df.to_excel(writer, sheet_name = f'{celltype} {target_name}'[:31], startrow = row_tracker)
                row_tracker += (len(values_df.index) + 2)
    workbook = load_workbook(Path(dir_output, out_name))
    if "Sheet1" in workbook.sheetnames:
        workbook.remove(workbook['Sheet1'])
        workbook.save(Path(dir_output, out_name))

In [ ]:
now = datetime.now()
print ("\n**********SCRIPT COMPLETE**********")
print(now.strftime("%Y-%m-%d_%H%M%S"))